In [1]:
import os 
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
MESHAPI_TOKEN = os.getenv("MESH_API_KEY")
MESH_BASE_URL = os.getenv("MESH_BASE_URL", "https://api.meshapi.ai")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

In [3]:
PINECONE_INDEX_NAME = "meshapi-demo-kb"
PINECONE_CLOUD="aws"
PINECONE_REGION="us-east-1"
EMBEDDING_DIMENSIONS = 1024

In [4]:
from meshapi import MeshAPI

client = MeshAPI(base_url=MESH_BASE_URL, token=MESHAPI_TOKEN)
print("Meshapi client ready")

Meshapi client ready


In [5]:
FAST_MODEL = "openai/gpt-4o-mini"
SMART_MODEL= "mistral/mistral-large-3-675b-instruct"
EMBEDDING_MODEL = "openai/text-embedding-3-small"

In [19]:
from meshapi import ChatCompletionParams, ChatMessage 

def ask(model, prompt, temperature=0.4, max_tokens=350):
    resp = client.chat.completions.create(
        ChatCompletionParams(
            model=model,
            messages=[ChatMessage(role="user", content=prompt)],
            temperature=temperature,
            max_tokens=max_tokens
        )
    )
    return resp.choices[0].message.content

In [7]:
from meshapi import EmbeddingsParams

def mesh_embed(text):
    resp = client.embeddings.create(
        EmbeddingsParams(
            model=EMBEDDING_MODEL,
            input=text,
            dimensions=EMBEDDING_DIMENSIONS
        )
    )
    return [d.embedding for d in sorted(resp.data, key=lambda d: d.index)]

In [8]:
vec = mesh_embed(["Hello world"])[0]
print(len(vec))

1024


In [8]:
#PINECONE
import time
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

if PINECONE_INDEX_NAME not in [idx["name"] for idx in pc.list_indexes()]:
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=EMBEDDING_DIMENSIONS,
        metric="cosine",
        spec=ServerlessSpec(cloud=PINECONE_CLOUD, region=PINECONE_REGION)
    )
    while not pc.describe_index(PINECONE_INDEX_NAME).status["ready"]:
        time.sleep(1)
        
index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())


DescribeIndexStatsResponse(dimension=1024, total_vector_count=0, metric='cosine', namespaces=0)


In [9]:
knowledge_base = [
    {"id": "doc-1", "title": "Refund Policy", "text": "Nimbus Cloud offers a 30-day money-back guarantee on all annual plans. Monthly plans can be cancelled anytime but are not eligible for partial refunds. Refund requests must be submitted through the billing portal within the eligibility window."},
    {"id": "doc-2", "title": "Storage Limits", "text": "The Starter plan includes 100GB of storage, Pro includes 2TB, and Enterprise is negotiated per contract. Exceeding your plan's limit pauses new uploads until you upgrade or free up space; existing files remain accessible."},
    {"id": "doc-3", "title": "Data Retention", "text": "Deleted files move to a Trash folder and are permanently removed after 30 days. Account cancellation triggers a 90-day data retention window before permanent deletion, during which reactivation restores all data."},
    {"id": "doc-4", "title": "Sharing & Permissions", "text": "Files can be shared via link (view or edit access) or invited by email with role-based permissions: Viewer, Commenter, Editor, Owner. Shared links can be password-protected and set to expire after a chosen number of days."},
    {"id": "doc-5", "title": "Two-Factor Authentication", "text": "2FA is optional for Starter and Pro plans but mandatory for all Enterprise accounts. Supported methods are authenticator apps (TOTP) and SMS. Recovery codes are generated once and shown only at setup time."},
    {"id": "doc-6", "title": "API Rate Limits", "text": "The Nimbus Cloud API allows 100 requests per minute on Starter, 1000 on Pro, and custom limits on Enterprise. Exceeding the limit returns HTTP 429 with a Retry-After header indicating when to resume."},
    {"id": "doc-7", "title": "Plan Downgrades", "text": "Downgrading takes effect at the end of the current billing cycle. If your stored data exceeds the new plan's limit, you'll have a 14-day grace period to remove files before uploads are paused."},
    {"id": "doc-8", "title": "Support Response Times", "text": "Starter plan support responds within 48 hours via email. Pro plan support responds within 24 hours and includes live chat. Enterprise customers get a dedicated support contact with a 4-hour SLA."},
]

In [10]:
def chunk_text(text, max_chars=500, overlap=50):
    chunks=[]
    start=0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

In [12]:
chunks = []
for doc in knowledge_base:
    for i, c in enumerate(chunk_text(doc["text"])):
        chunks.append({"doc_id": doc["id"], "title": doc["title"], "chunk_index": i, "text": c})

print(f"{len(chunks)} chunks from {len(knowledge_base)} documents.")

8 chunks from 8 documents.


In [13]:
embeddings = mesh_embed([c["text"] for c in chunks])

pinecone_vectors = [
    {
        "id": f"{c['doc_id']}-{c['chunk_index']}",
        "values": emb,
        "metadata": {"title": c["title"], "text": c["text"], "doc_id": c["doc_id"]},
    }
    for c, emb in zip(chunks, embeddings)
]

index.upsert(vectors=pinecone_vectors)
print(f"Upserted {len(pinecone_vectors)} chunks into Pinecone index '{PINECONE_INDEX_NAME}'.")

Upserted 8 chunks into Pinecone index 'meshapi-demo-kb'.


In [ ]:
#Vector Search
def retrieve(query, top_k=3):
    query_embedding = mesh_embed([query])[0]
    results = index.query(vector=query_embedding, top_k=top_k, include_metadata=True)
    return [
        {"score": m["score"], "title": m["metadata"]["title"], "text": m["metadata"]["text"]}
        for m in results["matches"]
    ]

for r in retrieve("How much storage do I get on the Pro plan?"):
    print(f"[{r['score']:.3f}] {r['title']}: {r['text'][:100]}...")

[0.687] Storage Limits: The Starter plan includes 100GB of storage, Pro includes 2TB, and Enterprise is negotiated per contr...
[0.455] Support Response Times: Starter plan support responds within 48 hours via email. Pro plan support responds within 24 hours a...
[0.365] Plan Downgrades: Downgrading takes effect at the end of the current billing cycle. If your stored data exceeds the ne...


In [20]:
def rag_answer(question, model=FAST_MODEL, top_k=3):
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(f"[{h['title']}] {h['text']}" for h in hits)
    prompt = f"""Answer the question using ONLY the context below. If the answer isn't in the context, say you don't know.

Context:
{context}

Question: {question}"""
    return ask(model, prompt, temperature=0.2, max_tokens=300), hits

answer, sources = rag_answer("What happens if I go over my storage limit?")
print(answer)
print("\nSources:", [s["title"] for s in sources])

If you exceed your plan's storage limit, new uploads will be paused until you upgrade or free up space, but existing files will remain accessible.

Sources: ['Storage Limits', 'Plan Downgrades', 'Data Retention']


In [22]:
#Multi-agent RAG
#3 agent: Researcher Agent, Writter Agent, Critic Agent

In [23]:
from langchain_openai import ChatOpenAI
MESHAPI_OPENAI_BASE_URL = f"{MESH_BASE_URL}/v1"

In [24]:
fast_chat = ChatOpenAI(
    base_url=MESHAPI_OPENAI_BASE_URL,
    api_key=MESHAPI_TOKEN,
    model_name=FAST_MODEL
)

smart_chat = ChatOpenAI(
    base_url=MESHAPI_OPENAI_BASE_URL,
    api_key=MESHAPI_TOKEN,
    model_name=SMART_MODEL
)

print("Langchain chat mode wired to MeshAPI:", FAST_MODEL, " | " ,SMART_MODEL)

Langchain chat mode wired to MeshAPI: openai/gpt-4o-mini  |  mistral/mistral-large-3-675b-instruct


In [25]:
import json
from langchain_core.tools import tool

@tool
def search_knowledge_base(query: str)->str:
    """Search the Nimbus cloud knowledge base for relevant policy/product information.
    Args:
        query: the search query text
    """
    hits = retrieve(query, top_k=3)
    return json.dumps(hits)

In [26]:
from langchain.agents import create_agent

researcher_agent = create_agent(
    model=fast_chat,
    tools=[search_knowledge_base],
    system_prompt=(
        "You are a research assistant. Use the search_knowledge_base tool to gather facts "
        "before answering. Once you have enough information, summarize the relevant facts as "
        "a short bullet list. Do not answer the user's question directly -- just report findings."
    ),
)

def run_researcher(question):
    result = researcher_agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

d:\py\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [27]:
writer_agent = create_agent(
    model=smart_chat,
    system_prompt=(
        "You are a helpful support agent for Nimbus Cloud. Using the research notes the user gives you, "
        "write a clear, friendly answer to their original question."
    ),
)

def run_writer(question, research_notes):
    prompt = f"Customer question: {question}\n\nResearch notes:\n{research_notes}"
    result = writer_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return result["messages"][-1].content

In [28]:
from pydantic import BaseModel, Field

class Critique(BaseModel):
    verdict: str = Field(description="'pass' or 'revise'")
    reason: str
    missing_info: str = Field(default="", description="what's missing, if verdict is 'revise'")

critic_agent = create_agent(
    model=smart_chat,
    system_prompt=(
        "You are a quality reviewer for customer support answers. Grade the draft strictly: does it "
        "fully and accurately answer the question using only reasonable support-agent knowledge?"
    ),
    response_format=Critique,
)

def run_critic(question, draft):
    prompt = f"Question: {question}\n\nDraft answer: {draft}\n\nGrade this draft."
    result = critic_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return result["structured_response"]

In [29]:
def research_write_review(question):
    print(f"Question: {question}\n")

    notes = run_researcher(question)
    print(f"--- Researcher ({FAST_MODEL}) ---\n{notes}\n")

    draft = run_writer(question, notes)
    print(f"--- Writer ({SMART_MODEL}) ---\n{draft}\n")

    critique = run_critic(question, draft)
    print(f"--- Critic ({SMART_MODEL}) --- verdict={critique.verdict}, reason={critique.reason}\n")

    return draft

In [30]:
final = research_write_review("If I cancel my monthly plan halfway through, do I get money back?")
print("\n=== FINAL ANSWER ===")
print(final)

Question: If I cancel my monthly plan halfway through, do I get money back?

--- Researcher (openai/gpt-4o-mini) ---
- Monthly plans can be canceled anytime, but they are not eligible for partial refunds.
- Refunds are only available for annual plans within a 30-day money-back guarantee.
- Cancellation requests must be made through the billing portal.

--- Writer (mistral/mistral-large-3-675b-instruct) ---
**Answer:**
Thanks for your question! If you cancel your **monthly Nimbus Cloud plan** partway through the billing cycle, we’re unable to offer a partial refund for the unused portion. Monthly plans are billed in full at the start of each cycle, and cancellations take effect at the end of that period.

However, if you’re on an **annual plan**, you can request a full refund within 30 days of purchase under our money-back guarantee. To cancel, just head to your **billing portal**—we’ll guide you through the process.

Let me know if you’d like help with anything else! 😊

--- Critic (mis

In [31]:
client.close()
print("Done.")

Done.
